# 02.1. Comparación de decoders

Ejecutar después de 02. Se usan las 50 preguntas oficiales y la evidencia guardada. Salamandra es el punto de partida recomendado que cumple el límite estricto. Todos los decoders del catálogo tienen licencia Apache 2.0 y hasta 8000 millones de parámetros.

El total sin RAGAS cubre 50 puntos. La corrección del texto libre aporta otros 30 y queda pendiente hasta ejecutar el juez oficial. El puntaje de 50 no basta para elegir por sí solo el decoder definitivo.

## Entorno

El mismo notebook corre en local o en Colab.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import zipfile

# Solo PyTorch: evita que transformers cargue TensorFlow/JAX de Colab
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"
try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
if MODO not in {"auto", "local", "colab"}:
    raise ValueError("MODO debe ser auto, local o colab")
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete_drive = DRIVE / "ai-week-colab.zip"
    if not paquete_drive.is_file():
        raise FileNotFoundError(f"Subir ai-week-colab.zip a {DRIVE}")
    paquete = Path("/content/ai-week-colab.zip")
    shutil.copy2(paquete_drive, paquete)
    huella_paquete = hash_archivo(paquete)
    raiz = Path("/content/aiweek") / huella_paquete[:12]
    if not (raiz / ".paquete_verificado").is_file():
        raiz.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(raiz)
        inventario = json.loads((raiz / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(raiz / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (raiz / ".paquete_verificado").write_text(huella_paquete)
    dependencias = raiz / "requirements-colab.txt"
    marca = Path("/content/aiweek_dependencias_v2.sha256")
    version_deps = hash_archivo(dependencias)
    if not marca.exists() or marca.read_text() != version_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        # Colab trae librerías compiladas para numpy 2; numpy 1.x rompe transformers (StringDType)
        if int(importlib.metadata.version("numpy").split(".")[0]) < 2:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2"], check=True)
        marca.write_text(version_deps)
    # llama.cpp: compilar solo para la GPU presente (L4 = 8.9) y con todos los núcleos
    capacidad = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip().splitlines()
    if capacidad:
        os.environ.setdefault("CUDAARCHS", capacidad[0].replace(".", ""))
    os.environ.setdefault("CMAKE_BUILD_PARALLEL_LEVEL", str(os.cpu_count()))
    for nombre in ("numpy", "pandas", "transformers", "tokenizers"):
        if nombre in sys.modules and getattr(sys.modules[nombre], "__version__", None) != importlib.metadata.version(nombre):
            raise RuntimeError("Reiniciar la sesión de Colab y ejecutar desde el principio para cargar las versiones instaladas")
    RESULTADOS = DRIVE / "resultados" / huella_paquete[:12]
else:
    raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/modelos.json").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("Abrir el notebook desde la carpeta del proyecto o notebooks")
    RESULTADOS = raiz / "data/experimentos_oficiales"
RAIZ = raiz
RESULTADOS.mkdir(parents=True, exist_ok=True)
if EN_COLAB:
    corpus_drive = RESULTADOS / "corpus_procesado.zip"
    marca_corpus = raiz / ".corpus_restaurado"
    if corpus_drive.is_file():
        huella_corpus = hash_archivo(corpus_drive)
        if not marca_corpus.exists() or marca_corpus.read_text() != huella_corpus:
            carpeta_corpus = raiz / "data/processed/corpus"
            with zipfile.ZipFile(corpus_drive) as z:
                for nombre in z.namelist():
                    if not (carpeta_corpus / nombre).resolve().is_relative_to(carpeta_corpus.resolve()):
                        raise ValueError("Ruta inválida en el corpus guardado")
                z.extractall(carpeta_corpus)
            marca_corpus.write_text(huella_corpus)
os.chdir(raiz)
sys.path.insert(0, str(raiz))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(raiz / "data/cache_huggingface") if EN_COLAB else os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Proyecto:", raiz)
print("Resultados:", RESULTADOS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno: Colab
Proyecto: /content/aiweek/915505e96337
Resultados: /content/drive/MyDrive/AIWEEK/resultados/915505e96337


In [2]:
import time
import platform
import pandas as pd
import matplotlib.pyplot as plt
from scripts.entorno.runtime import runtime, preparar_decoder, hardware, sha256_archivo
from scripts.evaluacion.oficial import (Evidencia, abrir_experimento, guardar_json, huellas,
                                      generar_oficial, evaluar_entrega)
from scripts.indice.recuperacion import hash_json
from scripts.evaluacion.comparar import comparar_corridas
from scripts.generacion.cliente import ServidorLocal
from scripts.generacion.comparar_decoders import Medidor

catalogo = json.loads((raiz / "configs/modelos.json").read_text(encoding="utf-8"))
puntero = json.loads((RESULTADOS / "recuperacion_actual.json").read_text(encoding="utf-8"))
carpeta_recuperacion = RESULTADOS / puntero["carpeta"]
archivo_recuperaciones = carpeta_recuperacion / "recuperaciones.json"
if sha256_archivo(archivo_recuperaciones) != puntero["sha256_recuperaciones"]:
    raise ValueError("Cambió la evidencia guardada")
if puntero["huellas"] != huellas(raiz):
    raise ValueError("Cambió el corpus, el código o el material oficial. Ejecutar 02 de nuevo")
recuperaciones = json.loads(archivo_recuperaciones.read_text(encoding="utf-8"))
if len(recuperaciones) != 50:
    raise ValueError("Ejecutar 02 para preparar las 50 preguntas")
evidencia = Evidencia(raiz, raiz / "data/processed/corpus")
print(hardware())

NVIDIA L4, 23034 MiB, 8.9


In [3]:
modelos = ['salamandra-7b-instruct', 'qwen3-4b', 'qwen2.5-7b-instruct', 'qwen2.5-1.5b-instruct']
etapa = 'decoders'
contexto = 8192
max_tokens = 1024
corridas = 2
reanudar = "20260927T214309068468Z"

## Preparación

Se usa Q4_K_M en todos los decoders para mantener comparable la cuantización. La L4 (22.5 GB) permite cargar cada modelo Q4_K_M completo en GPU, con contexto de 8192. Se registran las capas cargadas y la memoria observada.

La primera vez se compila llama.cpp en Colab y se descargan los pesos. Salamandra se convierte desde su revisión oficial y necesita 45 GiB libres durante la preparación. Si falta disco, se borran los encoders descargados en 01.1, que se pueden volver a descargar. Las siguientes ejecuciones dentro de la misma sesión reutilizan esos archivos.

In [4]:
fichas = [f for f in sorted(catalogo["decoders"], key=lambda f: f["prioridad"]) if f["nombre"] in modelos]
if len(fichas) != len(modelos) or len(set(modelos)) != len(modelos):
    raise ValueError("La selección contiene nombres desconocidos o repetidos")
if any(f["parametros"] > 8000000000 or f["licencia"] != "apache-2.0" for f in fichas):
    raise ValueError("Hay un modelo fuera del límite o de las licencias admitidas")
display(pd.DataFrame(fichas)[["nombre", "parametros", "licencia", "cuantizacion", "recomendado"]])
libre_gib = shutil.disk_usage(raiz).free / 2**30
print(f"Disco libre: {libre_gib:.1f} GiB")
servidor, fuente = runtime(raiz, catalogo)
pesos = {}
for ficha in fichas:
    print("Preparando", ficha["nombre"], flush=True)
    modelo = preparar_decoder(raiz, ficha, catalogo, servidor, fuente)
    archivos = [modelo] if ficha["nombre"] == "salamandra-7b-instruct" else [raiz / a["ruta"] for a in ficha["archivos"]]
    pesos[ficha["nombre"]] = {p.name: sha256_archivo(p) for p in archivos}
configuracion = {"modelos": modelos, "contexto": contexto, "max_tokens": max_tokens, "corridas": corridas,
                 "temperatura": 0, "semilla": 0, "gramatica": True, "cuantizacion": "Q4_K_M",
                 "sha256_recuperaciones": puntero["sha256_recuperaciones"], "pesos": pesos,
                 "sha256_servidor": sha256_archivo(servidor), "runtime_commit": catalogo["runtime"]["commit"],
                 "hardware": hardware(), "python": platform.python_version(), "huellas": huellas(raiz)}
carpeta = abrir_experimento(RESULTADOS, etapa, configuracion, reanudar)
print("Ejecución:", carpeta.name)

,nombre,parametros,licencia,cuantizacion,recomendado
0,salamandra-7b-instruct,7768117248,apache-2.0,Q4_K_M,True
1,qwen3-4b,4022468096,apache-2.0,Q4_K_M,False
2,qwen2.5-7b-instruct,7615616512,apache-2.0,Q4_K_M,False
3,qwen2.5-1.5b-instruct,1543714304,apache-2.0,Q4_K_M,False


Disco libre: 64.1 GiB
Preparando salamandra-7b-instruct
Verificado: config.json
Verificado: generation_config.json
model-00001-of-00004.safetensors: 49.0%
Verificado: model-00001-of-00004.safetensors
model-00002-of-00004.safetensors: 54.8%
Verificado: model-00002-of-00004.safetensors
model-00003-of-00004.safetensors: 77.2%
Verificado: model-00003-of-00004.safetensors
Verificado: model-00004-of-00004.safetensors
Verificado: model.safetensors.index.json
Verificado: special_tokens_map.json
Verificado: tokenizer.json
Verificado: tokenizer.model
Verificado: tokenizer_config.json
Preparando qwen3-4b
Qwen3-4B-Q4_K_M.gguf: 97.5%
Verificado: Qwen3-4B-Q4_K_M.gguf
Preparando qwen2.5-7b-instruct
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf: 6.9%
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf: 60.4%
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf: 91.7%
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf: 96.1%
Verificado: qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
Verificado: qwen2.5-7b-instruct-q

## Generación

Se guardan las respuestas originales y las salidas evaluadas por separado. Si falla el JSON o aparecen citas sin respaldo, la salida evaluada se abstiene y queda registrado el motivo. Los errores de ejecución detienen la celda y se pueden reintentar.

La ejecución 20260927T214309068468Z quedó interrumpida después de 202 respuestas registradas. reanudar apunta a esa carpeta. Al reconectar, ejecutar desde el inicio con la misma GPU L4 y los mismos parámetros. Las respuestas guardadas se omiten.

Cada pregunta terminada se guarda en Drive. Para continuar, copiar el nombre de la ejecución en reanudar. Se comprueba que la configuración coincida antes de reutilizar respuestas.

Todos reciben el mismo conjunto recuperado. Los tokenizadores pueden hacer que quepan distintos artículos. La tabla registra las omisiones y el archivo por_pregunta.csv permite comparar el contexto realmente utilizado.

In [ ]:
for ficha in fichas:
    nombre = ficha["nombre"]
    destino = carpeta / nombre
    destino.mkdir(exist_ok=True)
    pendientes = [(corrida, caso) for corrida in range(1, corridas + 1) for caso in recuperaciones
                  if not (destino / f"corrida_{corrida}" / f"{caso['entrada']['id']}.json").is_file()]
    if not pendientes:
        print(nombre, "ya está completo")
        continue
    with ServidorLocal(servidor, raiz / ficha["ruta"], destino, contexto=contexto) as motor:
        guardar_json(destino / "servidor.json", {"comando": motor.comando, "capas_gpu": motor.capas_gpu,
                                               "propiedades": motor.propiedades})
        comando = " ".join(map(str, motor.comando)) if isinstance(motor.comando, (list, tuple)) else str(motor.comando)
        print(nombre, "capas en GPU:", motor.capas_gpu, flush=True)
        print("Comando:", comando, flush=True)
        if not any(bandera in comando for bandera in ("-ngl", "--n-gpu-layers", "--gpu-layers")):
            print("AVISO: el comando no fija capas en GPU; llama.cpp podría estar corriendo en CPU", flush=True)
        primera = True
        for corrida, caso in pendientes:
            entrada = caso["entrada"]
            raw = destino / f"corrida_{corrida}" / "raw" / str(entrada["id"])
            inicio = time.perf_counter()
            with Medidor(motor.proceso.pid) as memoria:
                salida, registro = generar_oficial(motor.cliente, entrada, caso["pasajes"], evidencia, raw,
                                                   contexto=contexto, max_tokens=max_tokens)
            generacion_s = time.perf_counter() - inicio
            salida["latencia_ms"] = round(1000 * (generacion_s + caso["recuperacion_s"]))
            guardar_json(destino / f"corrida_{corrida}" / f"{entrada['id']}.json", {
                "salida": salida, "registro": registro, "generacion_s": generacion_s,
                "recuperacion_s": caso["recuperacion_s"], "memoria": memoria.resultado(),
            })
            print(nombre, corrida, entrada["id"], "abstención", salida["abstencion"],
                  f"{generacion_s:.1f} s", flush=True)
            if primera:
                primera = False
                print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                                      "--format=csv"], capture_output=True, text=True).stdout, flush=True)

salamandra-7b-instruct capas en GPU: {'cargadas': 33, 'total': 33}
Comando: /content/aiweek/915505e96337/data/runtime/fuente-2b129ccfa03aea330d2d9ac4650a10de393dbe3a/build/bin/llama-server --model /content/aiweek/915505e96337/data/modelos/salamandra-7b-instruct/salamandra-7b-instruct-Q4_K_M.gguf --host 127.0.0.1 --port 39235 --ctx-size 8192 --parallel 1 --seed 0 --temp 0 --repeat-penalty 1.0 --threads 8 --threads-batch 8 --n-gpu-layers auto --fit on --fit-target 512 --log-verbosity 4 --jinja --chat-template-kwargs {"enable_thinking":false}
salamandra-7b-instruct 1 51 abstención False 5.5 s
memory.used [MiB], memory.total [MiB], utilization.gpu [%]
5412 MiB, 23034 MiB, 4 %

salamandra-7b-instruct 1 58 abstención True 7.3 s
salamandra-7b-instruct 1 60 abstención True 7.7 s
salamandra-7b-instruct 1 128 abstención False 5.0 s
salamandra-7b-instruct 1 290 abstención False 6.8 s
salamandra-7b-instruct 1 308 abstención False 4.1 s
salamandra-7b-instruct 1 352 abstención True 26.3 s
salamandra

## Evaluación oficial

Se requieren las 50 respuestas para calcular el puntaje. La abstención cerrada con null se conserva y se registra como incompatibilidad del esquema publicado. No se considera una entrega totalmente validada mientras exista esa discrepancia.

La latencia guardada suma recuperación medida en 02 y generación de esta ejecución. Es una estimación con recuperación previamente calculada. El tiempo real de punta a punta debe medirse al integrar la aplicación.

In [ ]:
resumenes = []
detalles = []
for ficha in fichas:
    nombre = ficha["nombre"]
    respuestas_corridas = []
    registros_corridas = []
    for corrida in range(1, corridas + 1):
        destino = carpeta / nombre / f"corrida_{corrida}"
        registros = [json.loads((destino / f"{r['entrada']['id']}.json").read_text(encoding="utf-8")) for r in recuperaciones]
        respuestas = [r["salida"] for r in registros]
        reporte = evaluar_entrega(raiz, destino, respuestas)
        respuestas_corridas.append(respuestas)
        registros_corridas.append(registros)
        resumenes.append({"modelo": nombre, "corrida": corrida,
                          "cerradas_20": reporte["cerradas"]["puntos"], "citas_20": reporte["citas"]["puntos"],
                          "abstencion_10": reporte["abstencion"]["puntos"],
                          "total_50": reporte["total_automatico"]["obtenidos"],
                          "errores_esquema": reporte["errores_esquema"],
                          "abstenciones": sum(r["abstencion"] for r in respuestas),
                          "mediana_generacion_s": float(pd.Series([r["generacion_s"] for r in registros]).median()),
                          "vram_total_pico_mib": max((r["memoria"]["vram_total_pico_mib"] or 0 for r in registros)),
                          "respuestas_con_omisiones": sum(bool(r["registro"].get("omitidos")) for r in registros)})
        for r in registros:
            detalles.append({"modelo": nombre, "corrida": corrida, "id": r["salida"]["id"],
                             "formato": r["salida"]["formato"], "abstencion": r["salida"]["abstencion"],
                             "generacion_s": r["generacion_s"], "motivo": r["registro"].get("motivo"),
                             "fin": r["registro"].get("finish_reason"),
                             "pasajes_usados": len(r["salida"]["pasajes_recuperados"]),
                             "sha256_contexto": hash_json(r["salida"]["pasajes_recuperados"])})
    if corridas == 2:
        determinismo = comparar_corridas(*respuestas_corridas)
        determinismo["respuestas_raw_identicas"] = sum(a["registro"].get("contenido") == b["registro"].get("contenido")
                                                       for a, b in zip(*registros_corridas))
        guardar_json(carpeta / nombre / "determinismo.json", determinismo)
        print(nombre, determinismo)
tabla = pd.DataFrame(resumenes)
tabla.to_csv(carpeta / "resumen.csv", index=False)
pd.DataFrame(detalles).to_csv(carpeta / "por_pregunta.csv", index=False)
guardar_json(carpeta / "estado.json", {"estado": "completo", "modelos": modelos, "preguntas": 50, "corridas": corridas})
guardar_json(RESULTADOS / f"ultima_{etapa}.json", {"carpeta": str(carpeta.relative_to(RESULTADOS))})
display(tabla)
tabla[tabla.corrida == 1].set_index("modelo")[["cerradas_20", "citas_20", "abstencion_10"]].plot.barh(stacked=True, xlim=(0, 50))
plt.xlabel("Puntos automáticos sin RAGAS")
plt.tight_layout()
plt.show()

## Juez de texto libre

Opcional. Usa la llave entregada por los organizadores y consume su presupuesto de evaluación. En Colab se guarda en Secretos con el nombre OPENROUTER_API_KEY. Solo se pasa al evaluador oficial.

Ejecutar primero las mediciones sin API. Instalar las dependencias de RAGAS al final de la sesión para evitar que cambien el entorno usado durante la comparación.

In [ ]:
ejecutar_ragas = False
modelo_para_ragas = modelos[0]

if ejecutar_ragas:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(raiz / "data/oficial/scripts/requirements-evaluador.txt")], check=True)
    clave_anterior = os.environ.get("OPENROUTER_API_KEY")
    try:
        if EN_COLAB:
            from google.colab import userdata
            os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
        destino = carpeta / modelo_para_ragas / "corrida_1"
        respuestas = [json.loads((destino / f"{r['entrada']['id']}.json").read_text(encoding="utf-8"))["salida"] for r in recuperaciones]
        reporte_ragas = evaluar_entrega(raiz, destino, respuestas, ragas=True)
        display(pd.DataFrame([reporte_ragas["correccion_ragas"]]))
    finally:
        if clave_anterior is None:
            os.environ.pop("OPENROUTER_API_KEY", None)
        else:
            os.environ["OPENROUTER_API_KEY"] = clave_anterior


## Resultados y decisiones

Pendientes de esta ejecución. Revisar puntaje, errores por pregunta, abstenciones, omisiones y repetibilidad. Comparar la corrección de texto libre antes de fijar el decoder.

Las mediciones sobre la muestra sirven para desarrollo. La elección se registra después de leer los resultados. Los tiempos de la A100 no se trasladan al computador local.